# capa-recurrence-triage

Deviation records, investigation narratives, CAPA history and effectiveness-check
outcomes are a decade of labelled reasoning about root cause that no external
model has seen. It is the deepest proprietary text corpus most pharmaceutical
companies have, and no public version of it exists.

So this notebook runs on the closest public analogue: openFDA MAUDE device
reports, where a coded product problem stands in for a root cause and a recorded
remedial action stands in for a CAPA type. The analogy is structural and is
documented in `DATA_CARD.md`.

Nothing here dispositions anything. The models advise; the quality unit signs.

In [ ]:
import os, sys
from pathlib import Path
# make the notebook runnable from anywhere
HERE = Path.cwd()
if not (HERE / "run.py").exists():
    HERE = Path(globals().get("__vsc_ipynb_file__", ".")).resolve().parent
os.chdir(HERE)
sys.path.insert(0, str(HERE))
print("working in:", Path.cwd())

## 1. The data

With `SOURCE = "maude"` this is real openFDA device data, public domain under
CC0, and recurrence is derived rather than recorded. `DATA_CARD.md` gives the
inclusion rules, the group key and the censoring treatment.

`SOURCE = "synthetic"` switches to a generated corpus whose ids are prefixed
`DEV-SYN-`, so nothing in it can be mistaken for a real quality event. Two
signals are planted there deliberately: root cause is inferable from narrative
vocabulary, and recurrence depends on the action type. That corpus is for
learning the method. Do not quote a number from it.

In [ ]:
from src.store import load, temporal_split

SOURCE = "maude"   # "maude" for real data, "synthetic" to see the planted version
df = load(SOURCE)
print(f"{len(df)} records ({SOURCE}), {df.date.min().date()} -> {df.date.max().date()}\n")
print(df["cause_category"].value_counts().head(10).to_string())
print(f"\n365-day recurrence rate: {df.recurred_within_365d.mean():.1%}")

## 2. Start with the table that needs no model

Before any modelling, ask the data one question. Does the class of remedial
action recorded predict whether a comparable problem is reported again?

Read the answer, then read section 8, which is where it comes apart.

In [ ]:
from src.models import RecurrenceModel

table = RecurrenceModel().capa_risk_table(df, by="action_class")
table

In [ ]:
hi, lo = table.index[0], table.index[-1]
ratio = table.loc[hi, "recurrence_rate"] / table.loc[lo, "recurrence_rate"]
print(f"'{hi}' recurs {ratio:.1f}x as often as '{lo}'.")
print(f"and '{hi}' is the CAPA most often chosen for human-error root causes.")

That table is where most analyses of this kind stop. Section 8 is why this one
does not.

## 3. Evaluate on a temporal split

Train on the past, test on the future. A quality model evaluated on a random
split is being tested on events its own future already explained to it.

In [ ]:
from src.models import RootCauseClassifier

train, test = temporal_split(df)
print(f"train {len(train)} ({train.date.min().date()} -> {train.date.max().date()})")
print(f"test  {len(test)} ({test.date.min().date()} -> {test.date.max().date()})\n")

rc = RootCauseClassifier().fit(train)
print(rc.evaluate(test, train))

rm = RecurrenceModel().fit(train)
print(rm.evaluate(test, train))

## 4. What the classifier's accuracy actually measures

On the synthetic corpus, about 14% of non-human-error events are coded
`Human Error` anyway. That is the default a quality system falls back on when an
investigation runs out of time, and it sets the classifier's ceiling. It is a
real phenomenon, not a synthetic quirk, which is why it was planted.

On MAUDE the equivalent ceiling is set by how consistently manufacturers code
`product_problems`, which is unmeasured here. Either way the ceiling is the
coding, not the model. Measure it in your own data before promising anyone a
number.

In [ ]:
print(rc.report(test))

## 5. Precedent retrieval

What investigators actually want is not "what is the root cause" but *"has this happened here before, what did we do, and did it hold?"*

In [ ]:
from src.retrieve import PrecedentIndex, precedent_summary

idx = PrecedentIndex().fit(train)
hits = idx.search("alarm did not sound and the unit was removed from service",
                  k=5, same_site=train["unit"].mode().iat[0], same_area=train["area"].mode().iat[0])
hits[["record_id", "date", "cause_category", "action_type",
      "recurred_within_365d", "group_size", "similarity"]]

In [ ]:
precedent_summary(hits)

## 6. The full triage

Precedent, then root cause, then recommended action, then recurrence risk, then a
drafted narrative.

Watch the `**` line. It compares the action a quality unit is most likely to
choose against the one that historically held, and says so before the CAPA is
approved. Section 8 is the check on whether that comparison means anything.

In [ ]:
from src.triage import TriageSystem

ts = TriageSystem(train)
result = ts.triage(
    "the alarm did not sound during the procedure and the unit was removed from service",
    site=train["unit"].mode().iat[0], process_area=train["area"].mode().iat[0],
    severity=train["severity"].mode().iat[0], deviation_id="NEW-001")

print(result.render())

## 7. Prose instead of structure (optional)

The default composer uses no LLM. Set `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`, or
run Ollama locally, and pass a backend to get a written narrative instead.

The precedent block and the risk warning are appended either way, so the
reviewable content never depends on the generator.

In [ ]:
from src.llm import available_backends
available_backends()

# once a key is set:
# result = ts.triage(..., narrative_backend="anthropic")

## 8. The step that decides whether any of this means anything

On real data the table in section 2 is confounded. A device that gets reported
more is mechanically more likely to be reported again, whatever remedial action
was taken, and devices under regulatory attention attract both more reports and
more engineering-type actions. Reporting volume is a plausible common cause of
exposure and outcome.

The covariates have to be measured backwards. `group_size` counts every member of
a group including reports that arrive after the index record, so it encodes the
outcome. `n_prior_in_group` and `prior_365` count only the past.

Fit the association twice: once raw, once adjusted.

In [ ]:
from src.experiments import adjusted

adj = adjusted(SOURCE, n_boot=300)

The verdict line is printed rather than left to the reader. Whether the adjusted interval excludes 1 is what decides the claim, and on this corpus it does not: the adjusted odds ratio is 1.769 with a cluster-bootstrap 95% interval of [0.822, 5.166].

Note what that interval is not: tight. With 39 communication-class filing events, this corpus could not have detected anything short of a large effect.

## And the ablation shows where the signal actually lives

In [ ]:
from src.experiments import ablation

abl = ablation(SOURCE, seeds=3)

Watch the `group_size only` row. On the real corpus it reaches ROC-AUC 0.807 against 0.516 for the action type alone. A covariate that encodes nothing but how often a group gets reported should not beat every substantive feature, and the reason it does is that it counts events arriving after the index record. That is what identified the leak.

Note the direction, though. Here the leaky adjustment pulls the estimate *toward* the null (1.019 against a correct 1.769). On the earlier record-level corpus it pulled away from it. A leak has no consistent sign, which is exactly what makes it hard to notice.

The same contrast appears on the offline test fixture, which has no planted relationship at all: a correctly built pipeline fails its gate there, which is the intended behaviour.

## 9. Wiring in your QMS

`src/schema.py` is the contract. Most fields map straight across from TrackWise or
Veeva QMS. One does not.

In [ ]:
from src.schema import MAPPING
for k, v in MAPPING.items():
    print(f"{k:<28} {v}\n")

> `recurred_within_365d` is not a field in any QMS, and not a field in MAUDE
> either. You derive it: a later report in the same group within 365 days of this
> one.

That derivation is the asset, and it is also this study's main limitation. Three
of the recurrence model's five features are the group key that defines the
outcome, so its high AUC is circular rather than skilful. Build the linkage even
if you build nothing else here. Everything above depends on it, and it is a SQL
job rather than a modelling project.

## 10. Before this touches a regulated process

Write the context-of-use statement first, per FDA's risk-based credibility
framework. One page, at kickoff: the specific question the model answers, and the
consequence of it being wrong. It determines the entire validation burden and is
far cheaper to write before the model exists.

The trap is starting in the advisory band and drifting into decision support
without re-validating. Declare the band, and make a band change a formal
change.